## Частина 1. Робота з VHI-індексами (NOAA)

### Завдання 1-2: Створення віртуального середовища та автоматизація завантаження файлів за допомогою urllib
**Умова:** Для кожної з адміністративних одиниць України завантажити тестові структуровані файли, що містять значення VHI-індексу. При зберіганні файлу до його імені додати дату та час завантаження. Реалізувати механізм запобігання повторного довантаження та колізії даних. ID=0 (середнє по Україні) завантажувати не потрібно.

In [1]:
import os
import urllib.request
from datetime import datetime

target_dir = "../datasets/noaa"
os.makedirs(target_dir, exist_ok=True)

def download_vhi_data():
    print(f"--- Старт процесу перевірки та завантаження даних ---")
    
    for province_id in range(1, 28):
        
        already_downloaded = False
        for file in os.listdir(target_dir):
            if file.startswith(f"vhi_id_{province_id}_") and file.endswith(".csv"):
                print(f"Область №{province_id} вже завантажена раніше (файл: {file}). Пропускаємо.")
                already_downloaded = True
                break
                
        if already_downloaded:
            continue
            
        url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2=2024&type=Mean"
        
        try:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"{target_dir}/vhi_id_{province_id}_{timestamp}.csv"
            
            urllib.request.urlretrieve(url, filename)
            print(f"Успішно завантажено область №{province_id} -> {filename}")
        except Exception as e:
            print(f"Помилка при завантаженні області №{province_id}: {e}")
            
    print(f"--- Процес завершено ---")

download_vhi_data()

--- Старт процесу перевірки та завантаження даних ---
Успішно завантажено область №1 -> ../datasets/noaa/vhi_id_1_20260520_025108.csv
Успішно завантажено область №2 -> ../datasets/noaa/vhi_id_2_20260520_025123.csv
Успішно завантажено область №3 -> ../datasets/noaa/vhi_id_3_20260520_025129.csv
Успішно завантажено область №4 -> ../datasets/noaa/vhi_id_4_20260520_025132.csv
Успішно завантажено область №5 -> ../datasets/noaa/vhi_id_5_20260520_025142.csv
Успішно завантажено область №6 -> ../datasets/noaa/vhi_id_6_20260520_025143.csv
Успішно завантажено область №7 -> ../datasets/noaa/vhi_id_7_20260520_025153.csv
Успішно завантажено область №8 -> ../datasets/noaa/vhi_id_8_20260520_025154.csv
Успішно завантажено область №9 -> ../datasets/noaa/vhi_id_9_20260520_025155.csv
Успішно завантажено область №10 -> ../datasets/noaa/vhi_id_10_20260520_025157.csv
Успішно завантажено область №11 -> ../datasets/noaa/vhi_id_11_20260520_025158.csv
Успішно завантажено область №12 -> ../datasets/noaa/vhi_id_12_

### Завдання 3-4: Зчитування даних у Pandas DataFrame, Data Cleaning та реіндексація за українським алфавітом
**Умова:** Зчитати завантажені текстові файли у pandas dataframe. Здійснити data cleaning: прибрати зайві стовпці, заповнити пропуски, видалити зайвий текст тощо. Додати стовпчики з назвою та індексом області. Реалізувати процедуру зміни індексів: замінити індекси з NOAA (англійська абетка) так, щоб області індексувалися за українською абеткою (1 область - Вінницька).

In [5]:
import pandas as pd
import glob
import re

# Словник відповідності нових ID (за укр. алфавітом) та назв областей
clean_nav_dict = {
    1: "Вінницька", 2: "Волинська", 3: "Дніпропетровська", 4: "Донецька",
    5: "Житомирська", 6: "Закарпатська", 7: "Запорізька", 8: "Івано-Франківська",
    9: "Київська", 10: "Кіровоградська", 11: "Луганська", 12: "Львівська",
    13: "Миколаївська", 14: "Одеська", 15: "Полтавська", 16: "Рівненська",
    17: "Сумська", 18: "Тернопільська", 19: "Харківська", 20: "Херсонська",
    21: "Хмельницька", 22: "Черкаська", 23: "Чернівецька", 24: "Чернігівська",
    25: "Республіка Крим", 26: "м. Севастополь", 27: "м. Київ"
}

# Карта переведення: оригінальний ID NOAA -> Новий ID за українською абеткою
noaa_to_ua_id = {
    1: 22, 2: 24, 3: 23, 4: 3, 5: 4, 6: 20, 7: 21, 8: 8, 9: 9, 10: 10,
    11: 11, 12: 12, 13: 13, 14: 14, 15: 15, 16: 16, 17: 17, 18: 18,
    19: 6, 20: 19, 21: 20, 22: 1, 23: 2, 24: 5, 25: 7, 26: 25, 27: 26
}

def load_and_clean_data(folder_path="../datasets/noaa"):
    all_frames = []
    file_paths = glob.glob(f"{folder_path}/vhi_id_*.csv")
    
    if not file_paths:
        print("Помилка: не знайдено CSV-файлів у папці ../datasets/noaa")
        return pd.DataFrame()
        
    for path in file_paths:
        match = re.search(r"vhi_id_(\d+)_", path)
        if not match:
            continue
        noaa_id = int(match.group(1))
        
        ua_id = noaa_to_ua_id.get(noaa_id, noaa_id)
        region_name = clean_nav_dict.get(ua_id, "Невідома область")
        
        try:
            with open(path, 'r') as f:
                lines = f.readlines()
        except Exception as e:
            print(f"Не вдалося прочитати файл {path}: {e}")
            continue
            
        clean_rows = []
        for line in lines:
            line_str = line.strip()
            # Пропускаємо теги HTML та пусті рядки
            if not line_str or line_str.startswith("<") or "html" in line_str.lower():
                continue
                
            # Видаляємо випадкові теги всередині рядка (буває в кінці файлу)
            line_str = re.sub(r'<.*?>', '', line_str)
            
            # Розбиваємо за комою
            parts = [p.strip() for p in line_str.split(',')]
            
            # Головна перевірка: перший елемент має бути роком (числом), 
            # і нам потрібні мінімум 7 колонок даних
            if parts[0].isnumeric() and len(parts) >= 7:
                # Беремо перші 7 значень (Year, Week, SMN, SMT, VCI, TCI, VHI)
                clean_rows.append(parts[:7])
                
        if not clean_rows:
            continue
            
        # Створюємо датафрейм для поточної області
        df = pd.DataFrame(clean_rows, columns=['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI'])
        
        # Перетворюємо типи в числові
        df['Year'] = df['Year'].astype(int)
        df['Week'] = df['Week'].astype(int)
        df['VHI'] = pd.to_numeric(df['VHI'], errors='coerce')
        
        # Обробка missing data за ТЗ (видаляємо аномальні -1 та NaN)
        df = df.dropna(subset=['VHI'])
        df = df[df['VHI'] >= 0]
        
        # Додаємо ідентифікатори
        df['Area_ID'] = ua_id
        df['Area_Name'] = region_name
        
        # Залишаємо лише потрібні колонки
        df = df[['Area_ID', 'Area_Name', 'Year', 'Week', 'VHI']]
        all_frames.append(df)
        
    if not all_frames:
        print("Помилка: Не вдалося розпарсити жодного файлу. Перевір вміст папки datasets.")
        return pd.DataFrame()
        
    # Об'єднуємо все разом
    main_df = pd.concat(all_frames, ignore_index=True)
    main_df = main_df.sort_values(by=['Area_ID', 'Year', 'Week']).reset_index(drop=True)
    return main_df

# Запуск
vhi_df = load_and_clean_data()
print("--- Дані УСПІШНО оброблені та завантажені! ---")
print(f"Загальна кількість записів у таблиці: {vhi_df.shape[0]}")
vhi_df.head(15)

--- Дані УСПІШНО оброблені та завантажені! ---
Загальна кількість записів у таблиці: 58995


,Area_ID,Area_Name,Year,Week,VHI
0,1,Вінницька,1982,2,45.92
1,1,Вінницька,1982,3,43.50
2,1,Вінницька,1982,4,39.12
3,1,Вінницька,1982,5,35.60
4,1,Вінницька,1982,6,33.57
5,1,Вінницька,1982,7,33.20
6,1,Вінницька,1982,8,32.73
7,1,Вінницька,1982,9,32.80
8,1,Вінницька,1982,10,32.46
9,1,Вінницька,1982,11,31.02


### Завдання 5: Процедури пошуку екстремумів та фільтрації даних
**Умова:** Реалізувати наступні функції для роботи з отриманим DataFrame:
1. Ряд VHI для заданого року та заданої області.
2. Пошук екстремумів (мінімум та максимум) VHI для заданого року та області.
3. Вибірка років, протягом яких спостерігався екстремальний посушливий період (VHI < 15) для вказаного відсотка областей або для конкретної області.

In [11]:
def get_vhi_series(df, year, area_id):
    """Повертає фільтрований DataFrame або Series із тижнями та значеннями VHI"""
    result = df.loc[(df['Year'] == year) & (df['Area_ID'] == area_id), ['Week', 'VHI']]
    return result.reset_index(drop=True)
    
print("1. Тест вибірки ряду VHI для області №1 за 2020 рік (перші 5 рядків):")
display(get_vhi_series(vhi_df, year=2020, area_id=1).head())

1. Тест вибірки ряду VHI для області №1 за 2020 рік (перші 5 рядків):


,Week,VHI
0,1,41.75
1,2,43.90
2,3,45.12
3,4,45.32
4,5,44.78


In [14]:
def get_vhi_extremes(df, year, area_id):
    """Знаходить мінімальне та максимальне значення VHI та тижні, коли вони сталися"""
    sub_df = df.loc[(df['Year'] == year) & (df['Area_ID'] == area_id)]
    
    if sub_df.empty:
        return None
        
    min_row = sub_df.loc[sub_df['VHI'].idxmin()]
    max_row = sub_df.loc[sub_df['VHI'].idxmax()]
    
    extremes = {
        'Min_VHI': min_row['VHI'], 'Min_Week': int(min_row['Week']),
        'Max_VHI': max_row['VHI'], 'Max_Week': int(max_row['Week'])
    }
    return extremes

print("2. Тест пошуку екстремумів для області №1 за 2020 рік:")
extremes = get_vhi_extremes(vhi_df, year=2020, area_id=1)

if extremes:
    print(f"  Мінімум VHI: {extremes['Min_VHI']} (Тиждень {extremes['Min_Week']})")
    print(f"  Максимум VHI: {extremes['Max_VHI']} (Тиждень {extremes['Max_Week']})")
else:
    print("  Дані за вказаний рік або область відсутні.")

2. Тест пошуку екстремумів для області №1 за 2020 рік:
  Мінімум VHI: 40.2 (Тиждень 16)
  Максимум VHI: 68.55 (Тиждень 29)


In [17]:
def get_extreme_drought_years(df, area_id=None, threshold_pct=None):
    """
    Якщо задано area_id: роки, де в області був хоча б один тиждень з VHI < 15.
    Якщо задано threshold_pct: роки, де більше ніж X% областей України мали VHI < 15.
    """
    drought_df = df[df['VHI'] < 15]
    
    if area_id is not None:
        years = drought_df.loc[drought_df['Area_ID'] == area_id, 'Year'].unique()
        return sorted(list(years))
        
    if threshold_pct is not None:
        total_areas = len(df['Area_ID'].unique())
        drought_years = []
        
        for year, group in df.groupby('Year'):
            areas_with_drought = group[group['VHI'] < 15]['Area_ID'].nunique()
            pct = (areas_with_drought / total_areas) * 100
            
            if pct > threshold_pct:
                drought_years.append((year, round(pct, 2)))
                
        return drought_years

print("3. Роки, коли в області №5 спостерігалася екстремальна посуха (VHI < 15):")
print(get_extreme_drought_years(vhi_df, area_id=5))

3. Роки, коли в області №1 спостерігалася екстремальна посуха (VHI < 15):
[np.int64(2000)]
